# LLM Benchmark — Claude · GPT · Gemini · Llama on Databricks
### Fully self-contained notebook · no external modules required

**What this does:**
- Sends 13 identical prompts to **14 models** across 4 providers via Databricks AI Gateway
- Providers: Anthropic Claude · OpenAI GPT · Google Gemini · Meta Llama
- Auto-validates every response (code execution · JSON · regex · SQL · math · classification)
- Reports: tokens · vendor AI cost · response time · accuracy · % diff from mean

> **Note:** Llama models are open-source — vendor AI cost = $0. Databricks charges DBU compute only for those.

**Before running:** fill in your credentials in Cell 2, then `Run All`.

---
## Cell 1 · Install dependencies

In [ ]:
%pip install openai pandas matplotlib -q

---
## Cell 2 · Credentials  ← fill these in

In [ ]:
import os

DATABRICKS_HOST  = "https://adb-<workspace-id>.azuredatabricks.net"  # ← your workspace URL
DATABRICKS_TOKEN = "dapi<your-personal-access-token>"                 # ← your PAT

os.environ["DATABRICKS_HOST"]  = DATABRICKS_HOST
os.environ["DATABRICKS_TOKEN"] = DATABRICKS_TOKEN

print("Host :", DATABRICKS_HOST)
print("Token:", DATABRICKS_TOKEN[:8] + "…")

---
## Cell 3 · Model registry & pricing

In [ ]:
import pandas as pd

# ── Model registry ─────────────────────────────────────────────────────────
# Model IDs use the databricks- prefix (Databricks Playground / Foundation Model APIs)
# ALL prices are vendor/market token rates in USD per 1M tokens — no DBU costs.
# Meta Llama pricing based on Meta Llama API / OpenRouter market rates.

MODELS = [
    # ── Anthropic Claude ──────────────────────────────────────────────────
    {"model_id": "databricks-claude-opus-4-7",             "display_name": "Claude Opus 4.7",        "provider": "Anthropic", "input_price_per_1m": 15.00, "output_price_per_1m": 75.00},
    {"model_id": "databricks-claude-sonnet-4-6",           "display_name": "Claude Sonnet 4.6",      "provider": "Anthropic", "input_price_per_1m":  3.00, "output_price_per_1m": 15.00},
    {"model_id": "databricks-claude-sonnet-4-5",           "display_name": "Claude Sonnet 4.5",      "provider": "Anthropic", "input_price_per_1m":  3.00, "output_price_per_1m": 15.00},
    {"model_id": "databricks-claude-haiku-4-5",            "display_name": "Claude Haiku 4.5",       "provider": "Anthropic", "input_price_per_1m":  0.80, "output_price_per_1m":  4.00},

    # ── OpenAI GPT ────────────────────────────────────────────────────────
    {"model_id": "databricks-gpt-4o",                      "display_name": "GPT-4o",                 "provider": "OpenAI",    "input_price_per_1m":  2.50, "output_price_per_1m": 10.00},
    {"model_id": "databricks-gpt-4o-mini",                 "display_name": "GPT-4o-mini",            "provider": "OpenAI",    "input_price_per_1m":  0.15, "output_price_per_1m":  0.60},

    # ── Google Gemini ─────────────────────────────────────────────────────
    {"model_id": "databricks-gemini-3-pro",                "display_name": "Gemini 3 Pro",           "provider": "Google",    "input_price_per_1m":  2.50, "output_price_per_1m": 10.00},
    {"model_id": "databricks-gemini-3-flash",              "display_name": "Gemini 3 Flash",         "provider": "Google",    "input_price_per_1m":  0.10, "output_price_per_1m":  0.40},
    {"model_id": "databricks-gemini-2-5-pro",              "display_name": "Gemini 2.5 Pro",         "provider": "Google",    "input_price_per_1m":  1.25, "output_price_per_1m": 10.00},
    {"model_id": "databricks-gemini-2-5-flash",            "display_name": "Gemini 2.5 Flash",       "provider": "Google",    "input_price_per_1m":  0.075,"output_price_per_1m":  0.30},

    # ── Meta Llama (pricing: Meta Llama API / OpenRouter market rates) ────
    {"model_id": "databricks-llama-4-maverick",            "display_name": "Llama 4 Maverick",       "provider": "Meta",      "input_price_per_1m":  0.15, "output_price_per_1m":  0.60},
    {"model_id": "databricks-meta-llama-3-3-70b-instruct", "display_name": "Llama 3.3 70B Instruct", "provider": "Meta",      "input_price_per_1m":  0.12, "output_price_per_1m":  0.38},
    {"model_id": "databricks-meta-llama-3-1-405b-instruct","display_name": "Llama 3.1 405B Instruct","provider": "Meta",      "input_price_per_1m":  4.00, "output_price_per_1m":  4.00},
    {"model_id": "databricks-meta-llama-3-1-8b-instruct",  "display_name": "Llama 3.1 8B Instruct",  "provider": "Meta",      "input_price_per_1m":  0.02, "output_price_per_1m":  0.05},
]

PRICING = {m["model_id"]: m for m in MODELS}

# ── Preview pricing table ──────────────────────────────────────────────────
df_models = pd.DataFrame(MODELS)[["display_name","provider","input_price_per_1m","output_price_per_1m"]] \
    .rename(columns={"display_name":"Model","provider":"Provider",
                     "input_price_per_1m":"Input $/1M tokens","output_price_per_1m":"Output $/1M tokens"})

df_models.style \
    .format({"Input $/1M tokens": "${:.3f}", "Output $/1M tokens": "${:.3f}"}) \
    .background_gradient(subset=["Input $/1M tokens","Output $/1M tokens"], cmap="RdYlGn_r") \
    .set_caption(f"{len(MODELS)} models · 4 providers · all costs = vendor token rates (USD/1M tokens)") \
    .hide(axis="index")

---
## Cell 4 · Auto-validators

In [ ]:
import re, ast, json

def _tier(score):
    if score >= 1.0:  return "Excellent"
    if score >= 0.75: return "Good"
    if score >= 0.40: return "Partial"
    return "Poor"

def _extract_code(text):
    m = re.search(r"```(?:python)?\s*\n(.*?)```", text, re.DOTALL)
    return m.group(1).strip() if m else text.strip()

def _extract_json(text):
    m = re.search(r"(\{.*\}|\[.*\])", text, re.DOTALL)
    return json.loads(m.group(1)) if m else json.loads(text.strip())

# ── 1. Numeric ───────────────────────────────────────────────────────────────
def numeric_validator(expected, tolerance=0.01):
    def validate(r):
        nums = re.findall(r"-?\d+(?:\.\d+)?", r.replace(",", ""))
        if not nums: return "Poor", 0.0, "No number found"
        got = float(nums[0])
        diff = abs(got - expected)
        if diff <= tolerance:    return "Excellent", 1.0,  f"Correct: {got}"
        if diff <= tolerance*10: return "Partial",   0.40, f"Close: got {got}, expected {expected}"
        return "Poor", 0.0, f"Wrong: got {got}, expected {expected}"
    return validate

# ── 2. Code execution ────────────────────────────────────────────────────────
def code_exec_validator(test_cases):
    def validate(r):
        code = _extract_code(r)
        ns = {}
        try:
            exec(compile(ast.parse(code), "<llm>", "exec"), ns)
        except Exception as e:
            return "Poor", 0.0, f"Compile error: {e}"
        passed, fails = 0, []
        for tc in test_cases:
            try:
                result = eval(tc["call"], ns)
                if result == tc["expected"]: passed += 1
                else: fails.append(f"{tc['call']} → {result!r} (expected {tc['expected']!r})")
            except Exception as e:
                fails.append(f"{tc['call']} → ERROR: {e}")
        score = passed / len(test_cases)
        detail = f"{passed}/{len(test_cases)} passed" + (f". Failures: {'; '.join(fails)}" if fails else "")
        return _tier(score), round(score, 2), detail
    return validate

# ── 3. JSON schema ───────────────────────────────────────────────────────────
def json_schema_validator(required_keys, value_checks=None):
    def validate(r):
        try: data = _extract_json(r)
        except Exception as e: return "Poor", 0.0, f"Invalid JSON: {e}"
        if isinstance(data, list): data = data[0] if data else {}
        missing = [k for k in required_keys if k not in data]
        if missing:
            score = 1 - len(missing)/len(required_keys)
            return _tier(score), round(score,2), f"Missing keys: {missing}"
        fails = []
        for k, chk in (value_checks or {}).items():
            ok = chk(data.get(k)) if callable(chk) else data.get(k) == chk
            if not ok: fails.append(f"{k}={data.get(k)!r}")
        if fails:
            score = 1 - len(fails)/len(value_checks or {})
            return _tier(score), round(score,2), f"Value failures: {fails}"
        return "Excellent", 1.0, f"All {len(required_keys)} keys valid"
    return validate

# ── 4. Regex extraction ──────────────────────────────────────────────────────
def regex_extraction_validator(items_to_find, item_pattern, label_name="item"):
    def validate(r):
        found   = re.findall(item_pattern, r)
        invalid = [f for f in found if not re.fullmatch(item_pattern, f)]
        missing = [i for i in items_to_find if i.lower() not in r.lower()]
        if invalid: return "Partial", 0.40, f"Malformed {label_name}s: {invalid}"
        if missing:
            score = 1 - len(missing)/len(items_to_find)
            return _tier(score), round(score,2), f"Missing {label_name}s: {missing}"
        return "Excellent", 1.0, f"All {len(items_to_find)} {label_name}s found and valid"
    return validate

# ── 5. Classification ────────────────────────────────────────────────────────
def classification_validator(items, valid_labels, expected_mapping=None):
    def validate(r):
        text = r.lower()
        missing = [i for i in items if i.lower() not in text]
        if missing:
            score = 1 - len(missing)/len(items)
            return _tier(score), round(score,2), f"Items not classified: {missing}"
        if expected_mapping:
            wrong = [f"{item}→{lbl}" for item, lbl in expected_mapping.items()
                     if not re.search(re.escape(item.lower()) + r".*?" + re.escape(lbl.lower()), text, re.DOTALL)]
            if wrong:
                score = 1 - len(wrong)/len(expected_mapping)
                return _tier(score), round(score,2), f"Wrong labels: {wrong}"
        return "Excellent", 1.0, f"All {len(items)} items correctly classified"
    return validate

# ── 6. SQL ───────────────────────────────────────────────────────────────────
def sql_validator(required_clauses, forbidden_patterns=None):
    def validate(r):
        m = re.search(r"```(?:sql)?\s*\n(.*?)```", r, re.DOTALL | re.IGNORECASE)
        sql = m.group(1).strip() if m else r.strip()
        sql_up = sql.upper()
        bad = [p for p in (forbidden_patterns or []) if re.search(p, sql_up)]
        if bad: return "Poor", 0.0, f"Forbidden patterns: {bad}"
        missing = [c for c in required_clauses if c.upper() not in sql_up]
        if missing:
            score = 1 - len(missing)/len(required_clauses)
            return _tier(score), round(score,2), f"Missing clauses: {missing}"
        return "Excellent", 1.0, f"All {len(required_clauses)} clauses present"
    return validate

print("✓ Validators ready")

---
## Cell 5 · Test prompts (13 scenarios)

In [ ]:
TEST_PROMPTS = [
    # ── Factual ───────────────────────────────────────────────────────────────
    {
        "id": "capital_france", "category": "Factual",
        "prompt": "What is the capital city of France? Answer in one word only.",
        "validator": lambda r: ("Excellent", 1.0, "Correct") if "paris" in r.lower()
                               else ("Poor", 0.0, f"Expected Paris, got: {r.strip()[:30]}"),
    },

    # ── Math — prompts explicitly ask for a bare number only ──────────────────
    {
        "id": "compound_interest", "category": "Math",
        "prompt": (
            "Calculate compound interest: principal=$10,000, annual rate=5%, "
            "compounded monthly, time=3 years.\n"
            "Rules: return ONLY the final dollar amount as a number with 2 decimal places. "
            "No words, no formula, no dollar sign. Example output: 11614.72"
        ),
        "validator": numeric_validator(expected=11614.72, tolerance=0.05),
    },
    {
        "id": "prime_count", "category": "Math",
        "prompt": (
            "How many prime numbers are there between 1 and 50?\n"
            "Rules: return ONLY a single integer. No words or explanation. Example output: 15"
        ),
        "validator": numeric_validator(expected=15, tolerance=0),
    },

    # ── Code ──────────────────────────────────────────────────────────────────
    {
        "id": "code_reverse_string", "category": "Code",
        "prompt": "Write a Python function named `reverse_string` that takes a string and returns it reversed. Return only the function, no explanation.",
        "validator": code_exec_validator([
            {"call": "reverse_string('hello')",      "expected": "olleh"},
            {"call": "reverse_string('databricks')", "expected": "skcirbaatad"},
            {"call": "reverse_string('')",            "expected": ""},
            {"call": "reverse_string('a')",           "expected": "a"},
        ]),
    },
    {
        "id": "code_fibonacci", "category": "Code",
        "prompt": "Write a Python function named `fibonacci` that returns the nth Fibonacci number (0-indexed: fibonacci(0)=0, fibonacci(1)=1). Return only the function.",
        "validator": code_exec_validator([
            {"call": "fibonacci(0)",  "expected": 0},
            {"call": "fibonacci(1)",  "expected": 1},
            {"call": "fibonacci(6)",  "expected": 8},
            {"call": "fibonacci(10)", "expected": 55},
        ]),
    },
    {
        "id": "code_word_count", "category": "Code",
        "prompt": "Write a Python function named `word_count` that takes a string and returns a dict of word frequencies (case-insensitive). Return only the function.",
        "validator": code_exec_validator([
            {"call": "word_count('the cat sat on the mat')", "expected": {"the":2,"cat":1,"sat":1,"on":1,"mat":1}},
            {"call": "word_count('Hello hello HELLO')",      "expected": {"hello":3}},
            {"call": "word_count('')",                        "expected": {}},
        ]),
    },

    # ── Data / JSON ───────────────────────────────────────────────────────────
    {
        "id": "json_person_extraction", "category": "Data",
        "prompt": (
            "Extract information from this text and return as JSON with fields: name, age, job_title, company.\n"
            "Text: 'John Smith is 34 years old and works as a Data Engineer at Databricks.'\n"
            "Return only the JSON object."
        ),
        "validator": json_schema_validator(
            required_keys=["name", "age", "job_title", "company"],
            value_checks={
                "name":      lambda v: "john" in str(v).lower(),
                "age":       lambda v: int(str(v).replace('"','')) == 34,
                "job_title": lambda v: "engineer" in str(v).lower(),
                "company":   lambda v: "databricks" in str(v).lower(),
            },
        ),
    },
    {
        "id": "json_product_list", "category": "Data",
        "prompt": "Return a JSON array of 3 products with fields: id (integer), name (string), price (float), in_stock (boolean). Return only the JSON array.",
        "validator": json_schema_validator(required_keys=["id", "name", "price", "in_stock"]),
    },

    # ── Regex extraction ──────────────────────────────────────────────────────
    {
        "id": "extract_emails", "category": "Data",
        "prompt": (
            "Extract all email addresses from this text and list each on a new line:\n"
            "'Contact us at support@databricks.com or sales@company.org. For billing: billing@example.co.uk'"
        ),
        "validator": regex_extraction_validator(
            items_to_find=["support@databricks.com", "sales@company.org", "billing@example.co.uk"],
            item_pattern=r"[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}",
            label_name="email",
        ),
    },

    # ── Classification ────────────────────────────────────────────────────────
    {
        "id": "sentiment_multi", "category": "Classification",
        "prompt": (
            "Classify each review as Positive, Negative, or Neutral.\n"
            "1. 'Absolutely love this product!'\n"
            "2. 'Terrible experience, never buying again.'\n"
            "3. 'It arrived on time.'\n"
            "Format: '1. <label>, 2. <label>, 3. <label>'"
        ),
        "validator": classification_validator(
            items=["1", "2", "3"], valid_labels=["Positive", "Negative", "Neutral"],
            expected_mapping={"1": "Positive", "2": "Negative", "3": "Neutral"},
        ),
    },
    {
        "id": "topic_classification", "category": "Classification",
        "prompt": (
            "Classify each sentence into: Technology, Sports, Finance, or Health.\n"
            "A. 'The stock market hit a record high today.'\n"
            "B. 'Scientists developed a new cancer vaccine.'\n"
            "C. 'The team won the championship after overtime.'\n"
            "D. 'AI models are now used in chip design.'\n"
            "Format: 'A. <topic>, B. <topic>, C. <topic>, D. <topic>'"
        ),
        "validator": classification_validator(
            items=["A", "B", "C", "D"],
            valid_labels=["Technology", "Sports", "Finance", "Health"],
            expected_mapping={"A": "Finance", "B": "Health", "C": "Sports", "D": "Technology"},
        ),
    },

    # ── SQL ───────────────────────────────────────────────────────────────────
    {
        "id": "sql_aggregation", "category": "SQL",
        "prompt": (
            "Write a SQL query to find the top 3 customers by total order value "
            "from a table `orders(customer_id, order_amount, order_date)`. "
            "Include total_spent per customer. Return only the SQL."
        ),
        "validator": sql_validator(
            required_clauses=["SELECT", "FROM", "GROUP BY", "ORDER BY", "LIMIT"],
            forbidden_patterns=[r"SELECT \*", r"DROP", r"DELETE"],
        ),
    },
    {
        "id": "sql_window_function", "category": "SQL",
        "prompt": (
            "Write a SQL query using a window function to rank employees by salary "
            "within each department. Table: `employees(id, name, department, salary)`. "
            "Include: name, department, salary, rank. Return only the SQL."
        ),
        "validator": sql_validator(
            required_clauses=["SELECT", "FROM", "OVER", "PARTITION BY", "ORDER BY"],
            forbidden_patterns=[r"DROP", r"DELETE", r"TRUNCATE"],
        ),
    },
]

print(f"✓ {len(TEST_PROMPTS)} prompts loaded across categories: "
      f"{sorted(set(p['category'] for p in TEST_PROMPTS))}")

---
## Cell 6 · Select models to benchmark

In [ ]:
# ── Choose which providers to include ────────────────────────────────────────
# Available: "Anthropic"  "OpenAI"  "Google"  "Meta"
# Comment out any you don't want to run

SELECTED_PROVIDERS = [
    "Anthropic",   # Claude Opus 4.7 · Sonnet 4.6 · Sonnet 4.5 · Haiku 4.5
    "OpenAI",      # GPT-4o · GPT-4o-mini
    "Google",      # Gemini 3 Pro · Gemini 3 Flash · Gemini 2.5 Pro · Gemini 2.5 Flash
    "Meta",        # Llama 4 Maverick · Llama 3.3 70B · Llama 3.1 405B · Llama 3.1 8B
]

models_to_run = [m for m in MODELS if m["provider"] in SELECTED_PROVIDERS]
total_calls   = len(models_to_run) * len(TEST_PROMPTS)

print(f"{'Provider':<12} {'Model':<30} {'Input $/1M':>12} {'Output $/1M':>13}")
print("─" * 72)
for m in models_to_run:
    print(f"{m['provider']:<12} {m['display_name']:<30} ${m['input_price_per_1m']:>10.3f} ${m['output_price_per_1m']:>11.3f}")

print(f"\n→ {len(models_to_run)} models × {len(TEST_PROMPTS)} prompts = {total_calls} API calls")
print("  Cost calculated as: (input_tokens/1M × input_price) + (output_tokens/1M × output_price)")

---
## Cell 7 · Run benchmark  ⚡

In [ ]:
import openai, time
from dataclasses import dataclass
from typing import Optional

@dataclass
class Result:
    model_name:      str
    model_id:        str
    provider:        str
    prompt_id:       str
    prompt_category: str
    prompt_text:     str
    response_text:   str
    input_tokens:    int
    output_tokens:   int
    total_tokens:    int
    ai_cost_usd:     float
    response_time_s: float
    accuracy_label:  str
    accuracy_score:  float
    accuracy_detail: str
    error:           Optional[str] = None

def calc_cost(model_id, inp, out):
    p = PRICING.get(model_id)
    if not p: return 0.0
    return round((inp / 1_000_000) * p["input_price_per_1m"] +
                 (out / 1_000_000) * p["output_price_per_1m"], 8)

# ── Rate-limit retry with exponential back-off ────────────────────────────────
def call_with_retry(client, model_id, prompt_text, max_tokens=512,
                    max_retries=4, base_wait=2):
    """
    Calls the model and retries on 429 (rate limit) with exponential back-off.
    Waits: 2s → 4s → 8s → 16s before giving up.
    """
    for attempt in range(max_retries):
        try:
            return client.chat.completions.create(
                model=model_id,
                messages=[{"role": "user", "content": prompt_text}],
                max_tokens=max_tokens,
                temperature=0,
            )
        except openai.RateLimitError:
            wait = base_wait * (2 ** attempt)
            if attempt < max_retries - 1:
                print(f"    ⏳ 429 rate limit — waiting {wait}s before retry {attempt+2}/{max_retries}...", flush=True)
                time.sleep(wait)
            else:
                raise   # exhausted retries

# ── Build client ──────────────────────────────────────────────────────────────
client = openai.OpenAI(
    base_url=f"{DATABRICKS_HOST.rstrip('/')}/serving-endpoints",
    api_key=DATABRICKS_TOKEN,
)

# ── Run benchmark ─────────────────────────────────────────────────────────────
# CALL_DELAY: seconds to sleep between every API call (reduce 429s on low-quota workspaces)
CALL_DELAY = 1.0

results  = []
total    = len(models_to_run) * len(TEST_PROMPTS)
count    = 0

for model in models_to_run:
    for prompt in TEST_PROMPTS:
        count += 1
        print(f"[{count:>2}/{total}] {model['display_name']:<28} ← {prompt['id']}", end="  ", flush=True)
        start = time.perf_counter()

        try:
            resp = call_with_retry(client, model["model_id"], prompt["prompt"])
            elapsed        = round(time.perf_counter() - start, 2)
            response_text  = resp.choices[0].message.content or ""
            input_tokens   = resp.usage.prompt_tokens     if resp.usage else 0
            output_tokens  = resp.usage.completion_tokens if resp.usage else 0
            total_tokens   = resp.usage.total_tokens      if resp.usage else 0
            ai_cost        = calc_cost(model["model_id"], input_tokens, output_tokens)
            label, score, detail = prompt["validator"](response_text)
            print(f"→ {label:<10}  score={score:.2f}  {elapsed:.2f}s  ${ai_cost:.6f}")
            error = None

        except Exception as e:
            elapsed       = round(time.perf_counter() - start, 2)
            response_text = ""
            input_tokens = output_tokens = total_tokens = 0
            ai_cost      = 0.0
            label, score, detail = "Error", 0.0, str(e)
            error        = str(e)
            print(f"→ ERROR: {str(e)[:70]}")

        results.append(Result(
            model_name=model["display_name"], model_id=model["model_id"],
            provider=model["provider"],
            prompt_id=prompt["id"], prompt_category=prompt["category"],
            prompt_text=prompt["prompt"], response_text=response_text,
            input_tokens=input_tokens, output_tokens=output_tokens,
            total_tokens=total_tokens, ai_cost_usd=ai_cost,
            response_time_s=elapsed,
            accuracy_label=label, accuracy_score=score,
            accuracy_detail=detail, error=error,
        ))

        time.sleep(CALL_DELAY)   # polite pause — prevents 429s on low-quota workspaces

print(f"\n✓ Done — {len(results)} results  |  errors: {sum(1 for r in results if r.error)}")

---
## Cell 8 · Build DataFrame

In [ ]:
from dataclasses import asdict

df = pd.DataFrame([asdict(r) for r in results])
print(f"Shape: {df.shape}  |  Columns: {list(df.columns)}")

---
## Cell 9 · Detail results table

In [ ]:
ACCURACY_COLORS = {
    "Excellent": "background-color: #c6efce",
    "Good":      "background-color: #ffeb9c",
    "Partial":   "background-color: #ffcc99",
    "Poor":      "background-color: #ffc7ce",
    "Error":     "background-color: #d9d9d9",
}

df_detail = df[[
    "model_name", "prompt_id", "prompt_category",
    "input_tokens", "output_tokens", "total_tokens",
    "ai_cost_usd", "response_time_s",
    "accuracy_label", "accuracy_score", "accuracy_detail",
]].rename(columns={
    "model_name":      "Model",
    "prompt_id":       "Prompt",
    "prompt_category": "Category",
    "input_tokens":    "In Tok",
    "output_tokens":   "Out Tok",
    "total_tokens":    "Total Tok",
    "ai_cost_usd":     "AI Cost ($)",
    "response_time_s": "Time (s)",
    "accuracy_label":  "Result",
    "accuracy_score":  "Score",
    "accuracy_detail": "Validator Detail",
})

df_detail.style \
    .format({"AI Cost ($)": "${:.6f}", "Time (s)": "{:.2f}s", "Score": "{:.2f}"}) \
    .applymap(lambda v: ACCURACY_COLORS.get(v, ""), subset=["Result"]) \
    .set_caption("Detail: every model × every prompt — validator feedback in last column") \
    .hide(axis="index")

---
## Cell 10 · Summary with % difference from mean

In [ ]:
summary = (
    df.groupby("model_name")
    .agg(
        Prompts       = ("prompt_id",        "count"),
        Errors        = ("error",             lambda x: x.notna().sum()),
        Avg_Tokens    = ("total_tokens",      "mean"),
        Total_AI_Cost = ("ai_cost_usd",       "sum"),
        Avg_Time_sec  = ("response_time_s",   "mean"),
        Avg_Accuracy  = ("accuracy_score",    "mean"),
    )
    .reset_index()
    .rename(columns={"model_name": "Model"})
)

def pct_diff(series):
    mean = series.mean()
    return ((series - mean) / mean * 100).round(1) if mean != 0 else series * 0

summary["Cost_vs_Mean"]      = pct_diff(summary["Total_AI_Cost"])
summary["Time_vs_Mean"]      = pct_diff(summary["Avg_Time_sec"])
summary["Accuracy_vs_Mean"]  = pct_diff(summary["Avg_Accuracy"])
summary = summary.sort_values("Avg_Accuracy", ascending=False).reset_index(drop=True)

def fmt_pct(v):
    return f"+{v:.1f}%" if v >= 0 else f"{v:.1f}%"

summary.style \
    .format({
        "Avg_Tokens":       "{:.0f}",
        "Total_AI_Cost":    "${:.6f}",
        "Avg_Time_sec":     "{:.2f}s",
        "Avg_Accuracy":     "{:.2f}",
        "Cost_vs_Mean":     fmt_pct,
        "Time_vs_Mean":     fmt_pct,
        "Accuracy_vs_Mean": fmt_pct,
    }) \
    .background_gradient(subset=["Avg_Accuracy"],  cmap="RdYlGn") \
    .background_gradient(subset=["Total_AI_Cost"], cmap="RdYlGn_r") \
    .background_gradient(subset=["Avg_Time_sec"],  cmap="RdYlGn_r") \
    .applymap(lambda v: "color:green;font-weight:bold" if v>=0 else "color:red;font-weight:bold",
              subset=["Accuracy_vs_Mean"]) \
    .applymap(lambda v: "color:red;font-weight:bold" if v>=0 else "color:green;font-weight:bold",
              subset=["Cost_vs_Mean", "Time_vs_Mean"]) \
    .set_caption("Summary — sorted by accuracy · % diff = deviation from group mean") \
    .hide(axis="index")

---
## Cell 11 · Charts — accuracy · cost · response time

In [ ]:
import matplotlib.pyplot as plt

COLORS = ["#4472C4","#ED7D31","#A9D18E","#FFC000","#5B9BD5","#FF6B6B","#C55A11","#538135"]

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle("LLM Benchmark — Databricks AI Gateway", fontsize=14, fontweight="bold")

model_names = summary["Model"].tolist()
x = range(len(model_names))
bar_colors = COLORS[:len(model_names)]

specs = [
    (axes[0], "Avg_Accuracy",  "Avg Accuracy Score",    "Score (0–1)",  "{:.2f}",   0.0, 1.15),
    (axes[1], "Total_AI_Cost", "Total AI Cost (USD)",   "USD",          "${:.5f}",  None, None),
    (axes[2], "Avg_Time_sec",  "Avg Response Time (s)", "Seconds",      "{:.2f}s",  None, None),
]

for ax, col, title, ylabel, fmt, ymin, ymax in specs:
    vals = summary[col].tolist()
    bars = ax.bar(x, vals, color=bar_colors)
    ax.set_title(title, fontsize=11)
    ax.set_xticks(x)
    ax.set_xticklabels(model_names, rotation=35, ha="right", fontsize=8)
    ax.set_ylabel(ylabel)
    if ymin is not None: ax.set_ylim(ymin, ymax)
    mean_val = sum(vals)/len(vals)
    ax.axhline(mean_val, color="gray", linestyle="--", linewidth=1, label=f"Mean: {fmt.format(mean_val)}")
    ax.legend(fontsize=8)
    for i, v in enumerate(vals):
        ax.text(i, v * 1.03, fmt.format(v), ha="center", fontsize=8)

plt.tight_layout()
plt.savefig("llm_benchmark_charts.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → llm_benchmark_charts.png")

---
## Cell 12 · Accuracy heatmap by category × model

In [ ]:
pivot = df.pivot_table(
    index="prompt_category",
    columns="model_name",
    values="accuracy_score",
    aggfunc="mean",
)

pivot.style \
    .format("{:.2f}") \
    .background_gradient(cmap="RdYlGn", axis=None) \
    .set_caption("Avg accuracy by category × model  (green = best)")

---
## Cell 13 · Validator failure drill-down

In [ ]:
failures = df_detail[df_detail["Result"] != "Excellent"].copy()

if failures.empty:
    print("All responses scored Excellent!")
else:
    print(f"{len(failures)} non-Excellent responses:")
    failures.style \
        .format({"AI Cost ($)": "${:.6f}", "Time (s)": "{:.2f}s", "Score": "{:.2f}"}) \
        .applymap(lambda v: ACCURACY_COLORS.get(v, ""), subset=["Result"]) \
        .set_caption("Non-Excellent responses — 'Validator Detail' explains why") \
        .hide(axis="index")

---
## Cell 14 · Inspect any model + prompt

In [ ]:
# ── Change these to inspect any result ───────────────────────────────────────
# Model names:
#   Anthropic : Claude Opus 4.7 · Claude Sonnet 4.6 · Claude Sonnet 4.5 · Claude Haiku 4.5
#   OpenAI    : GPT-4o · GPT-4o-mini
#   Google    : Gemini 3 Pro · Gemini 3 Flash · Gemini 2.5 Pro · Gemini 2.5 Flash
#   Meta      : Llama 4 Maverick · Llama 3.3 70B Instruct · Llama 3.1 405B Instruct · Llama 3.1 8B Instruct
#
# Prompt IDs : capital_france · compound_interest · prime_count
#              code_reverse_string · code_fibonacci · code_word_count
#              json_person_extraction · json_product_list · extract_emails
#              sentiment_multi · topic_classification
#              sql_aggregation · sql_window_function

INSPECT_MODEL  = "Claude Sonnet 4.6"
INSPECT_PROMPT = "code_fibonacci"

row = df[(df["model_name"] == INSPECT_MODEL) & (df["prompt_id"] == INSPECT_PROMPT)]

if row.empty:
    print("No match — check INSPECT_MODEL and INSPECT_PROMPT.")
else:
    r = row.iloc[0]
    print(f"Model           : {r['model_name']}")
    print(f"Prompt          : {r['prompt_text']}")
    print(f"\n--- Response ---\n{r['response_text']}")
    print(f"\nTokens          : {r['input_tokens']} in / {r['output_tokens']} out")
    print(f"AI Cost         : ${r['ai_cost_usd']:.6f}" + (" (open-source, DBU only)" if r["ai_cost_usd"] == 0 else ""))
    print(f"Response Time   : {r['response_time_s']:.2f}s")
    print(f"Accuracy        : {r['accuracy_label']}  (score={r['accuracy_score']:.2f})")
    print(f"Validator Detail: {r['accuracy_detail']}")

---
## Cell 15 · Export to CSV

In [ ]:
csv_path = "llm_benchmark_results.csv"
df.drop(columns=["prompt_text", "response_text"]).to_csv(csv_path, index=False)
print(f"Saved {len(df)} rows → {csv_path}")